# Lab type: debug
# Course: ML301 — Deep Learning with PyTorch
# Lesson: Debugging Training Runs
# Task: The code below contains 3 bugs. All run without errors but produce wrong results. Find each bug, explain it in the markdown cell below it, and fix it.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# Synthetic house-price regression dataset
# Target: dollar values, mean ~250000, std ~80000
n = 600
X_raw = torch.randn(n, 10)
y_raw = 250_000 + 80_000 * (X_raw[:, :3].sum(dim=1) + 0.5 * torch.randn(n))

print(f'X shape: {X_raw.shape}')
print(f'y mean:  {y_raw.mean():.0f}')
print(f'y std:   {y_raw.std():.0f}')
print(f'y min:   {y_raw.min():.0f}')
print(f'y max:   {y_raw.max():.0f}')

## Bug 1: Sigmoid activation on a regression output

The network below is trained to predict house prices (targets in the range [0, 500000]). It uses `nn.Sigmoid()` as its final layer. MSELoss converges to a small-looking value, but all predictions are clustered near 0.0–1.0 instead of the target range.

Run the cell and observe the predicted values.

In [ ]:
class BuggyNet1(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),  # BUG: clamps output to [0, 1] — wrong for regression
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


dataset = TensorDataset(X_raw, y_raw)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)
criterion = nn.MSELoss()

model1 = BuggyNet1(10)
opt1 = torch.optim.Adam(model1.parameters(), lr=1e-3)

model1.train()
for epoch in range(5):
    epoch_loss = 0.0
    for Xb, yb in loader:
        opt1.zero_grad()
        loss = criterion(model1(Xb), yb)
        loss.backward()
        opt1.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch+1}  loss: {epoch_loss/len(loader):.2f}')

model1.eval()
with torch.no_grad():
    sample_preds = model1(X_raw[:8])
print('\nSample predictions:', sample_preds.tolist())
print('Sample targets:    ', y_raw[:8].tolist())

**Explain the bug:** Why does `nn.Sigmoid()` as the final layer make it impossible for this network to predict house prices correctly, even after many training epochs? What does the loss value tell you (and fail to tell you) in this scenario?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**Why Sigmoid kills regression:** `nn.Sigmoid()` maps any real-valued output to (0, 1). With targets of ~250 000, the network can never produce predictions in that range — the loss minimises to the best possible value within (0, 1), which is near zero. The loss *decreases* during training (appearing deceptively healthy) because the network converges to outputting ~0, but predictions remain ~100 000× too small. The loss value alone cannot reveal this; you must also check whether predicted values are in the expected range.

**Correct approach:** For regression over unbounded targets, use a linear output (no final activation). Only add a final non-linearity when the output range is genuinely constrained (e.g., Sigmoid for probabilities, Softmax for class distributions).

</details>

In [ ]:
# Fix: remove the Sigmoid — regression output should be unbounded
class FixedNet1(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            # No activation: linear output covers the full real line
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


torch.manual_seed(42)
model1_fixed = FixedNet1(10)
opt1_fixed   = torch.optim.Adam(model1_fixed.parameters(), lr=1e-3)

model1_fixed.train()
for epoch in range(5):
    epoch_loss = 0.0
    for Xb, yb in loader:
        opt1_fixed.zero_grad()
        loss = criterion(model1_fixed(Xb), yb)
        loss.backward()
        opt1_fixed.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch+1}  loss: {epoch_loss/len(loader):.2e}')

model1_fixed.eval()
with torch.no_grad():
    sample_preds = model1_fixed(X_raw[:8])
print('\nSample predictions:', [f'{v:.0f}' for v in sample_preds.tolist()])
print('Sample targets:    ', [f'{v:.0f}' for v in y_raw[:8].tolist()])

## Bug 2: MSELoss on unnormalised targets

The network architecture is correct (no Sigmoid). Targets are raw dollar values with mean ≈ 250 000 and std ≈ 80 000. MSELoss is computed on these raw values.

Run the cell and observe the loss magnitude and gradient behaviour.

In [ ]:
torch.manual_seed(42)
model2 = FixedNet1(10)
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

# BUG: MSELoss on raw dollar values — loss is ~10^10, gradients are enormous
raw_loader = DataLoader(TensorDataset(X_raw, y_raw), batch_size=64, shuffle=True)

model2.train()
for epoch in range(5):
    epoch_loss = 0.0
    for Xb, yb in raw_loader:
        opt2.zero_grad()
        loss = criterion(model2(Xb), yb)  # BUG: yb is unnormalised
        loss.backward()
        opt2.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch+1}  loss: {epoch_loss/len(raw_loader):.4e}')

print('\nWeight gradient norm (fc1):', model2.net[0].weight.grad.norm().item())

**Explain the bug:** MSELoss squares the residuals. With targets of magnitude ~250 000, what order of magnitude is the loss? How does this affect gradient updates and training stability? What is the fix, and why does normalising targets to zero mean / unit std resolve it?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**Loss magnitude problem:** MSELoss squares residuals, so with targets ~250 000 and predictions initially near zero, residuals are ~250 000 and the loss is ~6×10^10. Gradients scaled by such enormous values cause weight updates that are orders of magnitude too large, leading to unstable training, exploding parameters, or the optimizer overshooting and oscillating.

**Correct approach:** Normalise targets to zero mean and unit standard deviation before training. This brings the loss into the ~1–10 range where Adam's default `lr=1e-3` is well-calibrated. Inverse-transform (`pred * y_std + y_mean`) to convert predictions back to dollar scale for evaluation.

</details>

In [ ]:
# Fix: normalise targets to zero mean / unit std before training
y_mean = y_raw.mean()
y_std  = y_raw.std()
y_norm = (y_raw - y_mean) / y_std

print(f'Normalised y — mean: {y_norm.mean():.4f}, std: {y_norm.std():.4f}')

norm_loader = DataLoader(TensorDataset(X_raw, y_norm), batch_size=64, shuffle=True)

torch.manual_seed(42)
model2_fixed = FixedNet1(10)
opt2_fixed   = torch.optim.Adam(model2_fixed.parameters(), lr=1e-3)

model2_fixed.train()
for epoch in range(5):
    epoch_loss = 0.0
    for Xb, yb in norm_loader:
        opt2_fixed.zero_grad()
        loss = criterion(model2_fixed(Xb), yb)
        loss.backward()
        opt2_fixed.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch+1}  loss: {epoch_loss/len(norm_loader):.4f}')

# Inverse-transform predictions to dollar scale for evaluation
model2_fixed.eval()
with torch.no_grad():
    preds_norm = model2_fixed(X_raw[:8])
preds_dollars = preds_norm * y_std + y_mean
print('\nPredictions ($):', [f'{v:.0f}' for v in preds_dollars.tolist()])
print('Targets     ($):', [f'{v:.0f}' for v in y_raw[:8].tolist()])

## Bug 3: Adam learning rate 0.1

Architecture and target normalisation are both correct. The optimizer uses `lr=0.1`, which is 100× higher than the standard Adam starting point. Watch the loss across epochs.

In [ ]:
torch.manual_seed(42)
model3 = FixedNet1(10)

# BUG: lr=0.1 is far too high for Adam — loss oscillates or diverges
opt3 = torch.optim.Adam(model3.parameters(), lr=0.1)

model3.train()
for epoch in range(10):
    epoch_loss = 0.0
    for Xb, yb in norm_loader:
        opt3.zero_grad()
        loss = criterion(model3(Xb), yb)
        loss.backward()
        opt3.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch+1:2d}  loss: {epoch_loss/len(norm_loader):.4f}')

**Explain the bug:** Adam adapts per-parameter learning rates using gradient moment estimates. Why is a base learning rate of 0.1 problematic even with adaptive scaling? What convergence symptom does it produce, and what is the conventional starting point for Adam?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**Why lr=0.1 fails with Adam:** Adam's adaptive per-parameter scaling adjusts the *effective* step size based on gradient moment estimates, but it multiplies that adjustment by the global learning rate. A base `lr=0.1` is 100× the conventional starting point of `1e-3`, so all effective updates are 100× too large. The adaptive mechanism dampens large gradients but cannot compensate for a globally inflated lr — the result is loss oscillation or divergence after initial fast descent.

**Correct approach:** Use `lr=1e-3` as the default Adam starting point. Tune from there only after confirming training is stable; prefer learning-rate scheduling over raising the base rate.

</details>

In [ ]:
# Fix: use lr=1e-3 (standard Adam starting point)
torch.manual_seed(42)
model3_fixed = FixedNet1(10)
opt3_fixed   = torch.optim.Adam(model3_fixed.parameters(), lr=1e-3)

model3_fixed.train()
for epoch in range(10):
    epoch_loss = 0.0
    for Xb, yb in norm_loader:
        opt3_fixed.zero_grad()
        loss = criterion(model3_fixed(Xb), yb)
        loss.backward()
        opt3_fixed.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch+1:2d}  loss: {epoch_loss/len(norm_loader):.4f}')

## Summary

> **For each bug, write one sentence on what went wrong and how to catch it early.**

1. 
2. 
3. 

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Bug 1 — sigmoid on regression output:** `nn.Sigmoid()` clamps output to (0, 1), making it structurally impossible to predict targets in the range [0, 500 000] — remove the final activation for unbounded regression.

2. **Bug 2 — unnormalised MSE targets:** Squaring residuals of magnitude ~250 000 produces loss ~10^10 and enormous gradients; normalise targets to zero mean / unit std so loss and gradient scales are compatible with standard optimiser defaults.

3. **Bug 3 — Adam lr=0.1:** The global learning rate of 0.1 is 100× too high for Adam, overriding its adaptive scaling and causing oscillating loss — use the conventional starting point of `lr=1e-3`.

</details>